In [ ]:

import numpy as np
import os
import scipy
from load_data_function import load_data,save_data
import re
from load_data_function import fig_plot,battery_soh_plot,smooth_soh
import matplotlib.pyplot as plt
"""
XJTU_data为字典，keys为pachage_1, package_2, ..., package_6
每个package为字典，keys为battery_1, battery_2, ..., battery_n,其中第二个电池包中有15个电池，其余均为8个电池
每个battery为列表，列表中每个元素为numpy数组，shape为(3, n)，分别为电压、电流、时间，单位分别为V、A、s
"""


In [ ]:
#XJTU battery dataset
test_path= 'D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\XJTU battery dataset\Batch-1/2C_battery-1.mat'
data1= scipy.io.loadmat(test_path)


In [ ]:
print(data1.keys())

In [ ]:
print(data1['data'].shape)

In [ ]:
print(data1['data'][0][0][2])
print(data1['data'][0][0][2].shape)

In [ ]:
data2=data1['data'][0][0][2].T
print(data2.shape)
print(data2)


In [ ]:
def traverse_folders(root_dir):
    for dirpath, dirnames, filenames in os.walk(root_dir):
        # dirpath: 当前文件夹的绝对路径
        # dirnames: 当前文件夹下的子文件夹名列表
        # filenames: 当前文件夹下的文件名列表
        print("当前文件夹:", dirpath)
        print("子文件夹列表:", dirnames)
        print("文件列表:", filenames)
        print("----")

# 示例：遍历当前目录
traverse_folders("..")



In [ ]:

XJTU_path= 'D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\XJTU battery dataset'
def get_immediate_subdirs(parent_dir):
    # 获取所有条目名称，并过滤出子文件夹
    return [name for name in os.listdir(parent_dir)
            if os.path.isdir(os.path.join(parent_dir, name))]

package_list=get_immediate_subdirs(XJTU_path)
print(package_list)


XJTU_data={}
XJTU_SOH={}
for i,package in enumerate(package_list):
    if i==5:
        print(f'package: {package}')
        battery_list=os.listdir(os.path.join(XJTU_path,f'{package}'))
        battery_list=sorted(battery_list,key=lambda x: int(re.search(r'-(\d+)\.mat$', x).group(1)))
        print(battery_list)
        package1={}
        package2={}
        for j,battery in enumerate(battery_list):
            print(f'battery: {battery}')
            battery_path=os.path.join(XJTU_path,package,battery)
            data=scipy.io.loadmat(battery_path)

            voltage=[]
            current=[]
            time=[]
            capacity=[]
            package1[f'battery_{j+1}']=[]
            package2[f'battery_{j+1}']=[]
            for k in range(data['data'].shape[1]):
                voltage=data['data'][0][k][2].T
                current=data['data'][0][k][3].T
                time_segment = np.arange(0, voltage.shape[1],1)
                time=time_segment  # 转换为秒
                time=time.reshape(1,-1)

                package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))

            for k in range(data['data'].shape[1]-5):
                capacity_max1=np.max(data['data'][0][k][4].astype(float).T)
                capacity_max2=np.max(data['data'][0][k+1][4].astype(float).T)
                capacity_max3=np.max(data['data'][0][k+2][4].astype(float).T)
                capacity_max4=np.max(data['data'][0][k+3][4].astype(float).T)
                capacity_max5=np.max(data['data'][0][k+4][4].astype(float).T)
                capacity_max=np.max([capacity_max1,capacity_max2,capacity_max3,capacity_max4,capacity_max5])
                soh=capacity_max/2
                package2[f'battery_{j+1}'].append(soh)
            for k in range(data['data'].shape[1]-4,data['data'].shape[1]):
                soh=capacity_max/2
                package2[f'battery_{j+1}'].append(soh)
        XJTU_data[f'package_{i+1}']=package1
        XJTU_SOH[f'package_{i+1}']=package2
    else:

        print(f'package: {package}')
        battery_list=os.listdir(os.path.join(XJTU_path,f'{package}'))
        battery_list=sorted(battery_list,key=lambda x: int(re.search(r'-(\d+)\.mat$', x).group(1)))
        print(battery_list)

        package1={}
        package2={}
        for j,battery in enumerate(battery_list):
            print(f'battery: {battery}')
            battery_path=os.path.join(XJTU_path,package,battery)
            data=scipy.io.loadmat(battery_path)

            voltage=[]
            current=[]
            time=[]
            capacity=[]
            package1[f'battery_{j+1}']=[]
            package2[f'battery_{j+1}']=[]
            for k in range(data['data'].shape[1]):
                voltage=data['data'][0][k][2].T
                current=data['data'][0][k][3].T
                time_segment = np.arange(0, voltage.shape[1],1)
                time=time_segment  # 转换为秒
                time=time.reshape(1,-1)
                capacity=data['data'][0][k][4].T
                capacity_max=np.max(capacity)
                soh=capacity_max/2
                package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
                package2[f'battery_{j+1}'].append(soh)
        XJTU_data[f'package_{i+1}']=package1
        XJTU_SOH[f'package_{i+1}']=package2



In [ ]:
transformed_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data'
XJTU_path=os.path.join(transformed_path, 'XJTU_battery_dataset')
#save_data(XJTU_data, os.path.join(XJTU_path, 'XJTU_battery_data.pkl'))
XJTU_data=load_data(os.path.join(XJTU_path, 'XJTU_battery_data.pkl'))
#save_data(XJTU_SOH, os.path.join(XJTU_path, 'XJTU_battery_SOH.pkl'))
XJTU_SOH=load_data(os.path.join(XJTU_path, 'XJTU_battery_SOH.pkl'))

In [ ]:
print(XJTU_data.keys())

In [ ]:
print(XJTU_data['package_1'].keys())

In [ ]:
print(XJTU_data['package_1']['battery_1'])

In [ ]:
print(XJTU_data['package_1']['battery_1'][0])

In [ ]:
print(XJTU_data['package_1']['battery_1'][0].shape)

In [ ]:
fig_plot(XJTU_data['package_1']['battery_1'][0][0])

In [ ]:
fig_plot(XJTU_data['package_1']['battery_1'][1][1])

In [ ]:
fig_plot(XJTU_data['package_1']['battery_1'][2][2])

In [ ]:
print(XJTU_SOH.keys())

In [ ]:
print(XJTU_SOH['package_1'].keys())

In [ ]:
print(XJTU_SOH['package_6']['battery_1'])

In [ ]:
print(XJTU_SOH['package_1']['battery_1'][0])

In [ ]:
fig_plot(XJTU_SOH['package_1']['battery_1'][0:])

In [ ]:
plt.figure(figsize=(10,5))
package='package_1'
for key in XJTU_SOH[package].keys():
    plt.plot(XJTU_SOH[package][key])
plt.legend(list(XJTU_SOH[package].keys()))
plt.show()

In [ ]:
smooth_XJTU_SOH=smooth_soh(XJTU_SOH, method='moving_average', sigma=4)

In [ ]:
package='package_6'
battery_soh_plot(smooth_XJTU_SOH,XJTU_SOH[package].keys(),package)

In [ ]:
#save_data(XJTU_SOH, os.path.join(XJTU_path, 'XJTU_battery_SOH.pkl'))

In [ ]:
for package in XJTU_data.keys():
    for battery in XJTU_data[package].keys():
        if len(XJTU_data[package][battery])!= len(XJTU_SOH[package][battery]):
            print(f'battery {battery} has different length of data and SOH,data length: {len(XJTU_data[package][battery])}, SOH length: {len(XJTU_SOH[package][battery])}')

In [ ]:
for package in XJTU_data.keys():
    for battery in XJTU_data[package].keys():
        if len(XJTU_data[package][battery])!= len(XJTU_SOH[package][battery]):
            min_len=     min(len(XJTU_data[package][battery]),len(XJTU_SOH[package][battery]))
            #print(min_len)
            XJTU_data[package][battery]=XJTU_data[package][battery][:min_len][:][:]
            XJTU_SOH[package][battery]=XJTU_SOH[package][battery][:min_len][:][:]

In [ ]:
#save_data(XJTU_data,os.path.join(XJTU_path,'XJTU_battery_data.pkl'))
#save_data(XJTU_SOH,os.path.join(XJTU_path,'XJTU_battery_SOH.pkl'))
XJTU_data=load_data(os.path.join(XJTU_path,'XJTU_battery_data.pkl'))
XJTU_SOH=load_data(os.path.join(XJTU_path,'XJTU_battery_SOH.pkl'))

In [ ]:
for package in XJTU_data.keys():
    for battery in XJTU_data[package].keys():
        if len(XJTU_data[package][battery])!= len(XJTU_SOH[package][battery]):
            print(f'battery {battery} has different length of data and SOH,data length: {len(XJTU_data[package][battery])}, SOH length: {len(XJTU_SOH[package][battery])}')